In [1]:
import pandas as pd
import numpy as np

In [2]:
df_raw = pd.read_csv('kidney_disease.csv')

pd.set_option('display.max_columns', None)

df_raw

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,36.0,1.2,NaN,NaN,15.4,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,18.0,0.8,NaN,NaN,11.3,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,53.0,1.8,NaN,NaN,9.6,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,56.0,3.8,111.0,2.5,11.2,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,26.0,1.4,NaN,NaN,11.6,35,7300,4.6,no,no,no,good,no,no,ckd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,395,55.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,140.0,49.0,0.5,150.0,4.9,15.7,47,6700,4.9,no,no,no,good,no,no,notckd
396,396,42.0,70.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,75.0,31.0,1.2,141.0,3.5,16.5,54,7800,6.2,no,no,no,good,no,no,notckd
397,397,12.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,100.0,26.0,0.6,137.0,4.4,15.8,49,6600,5.4,no,no,no,good,no,no,notckd
398,398,17.0,60.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,114.0,50.0,1.0,135.0,4.9,14.2,51,7200,5.9,no,no,no,good,no,no,notckd


In [3]:
print(f'Shape: {df_raw.shape}  →  {df_raw.shape[0]} records, {df_raw.shape[1]} features\n')

# High-performance column inspection
col_info = pd.DataFrame({
    'pandas_dtype' : df_raw.dtypes.astype(str),
    'null_count'   : df_raw.isna().sum(),
    'null_%'       : (df_raw.isna().mean() * 100).round(2),
    'n_unique'     : df_raw.nunique(),
    'sample_values': [str(df_raw[c].dropna().unique()[:3].tolist()) for c in df_raw.columns]
}).sort_values('null_%', ascending=False)


def scan_data_quality(df):
    hidden_numeric = []
    dirty_categorical = []
    obj_df = df.select_dtypes(include=['object', 'string'])
    
    if obj_df.empty:
        return hidden_numeric, dirty_categorical
    
    for col in obj_df.columns:
        unique_vals = obj_df[col].dropna().unique()
        if unique_vals.size == 0:
            continue
            
        converted = pd.to_numeric(pd.Series(unique_vals), errors='coerce')
        is_probably_id = df[col].nunique() == len(df)
        
        if converted.notna().mean() > 0.5 and not is_probably_id:
            hidden_numeric.append(col)
            continue
            
        unique_series = pd.Series(unique_vals).astype(str)
        if unique_series.size != unique_series.str.strip().nunique():
            dirty_categorical.append((col, unique_vals.tolist()))
            
    return hidden_numeric, dirty_categorical


# Execute data scan
hidden_numeric_cols, dirty_categorical_cols = scan_data_quality(df_raw)
dirty_cols_names = [item[0] for item in dirty_categorical_cols]

# Define the row-by-row styling function for the DataFrame
def style_anomalies(row):
    color_numeric = 'background-color: #ffcccc'      # Soft red/coral
    color_categorical = 'background-color: #fff2cc'  # Soft yellow/gold
    color_default = ''                                # Standard transparent
    
    if row.name in hidden_numeric_cols:
        return [color_numeric] * len(row)
    elif row.name in dirty_cols_names:
        return [color_categorical] * len(row)
    return [color_default] * len(row)

# FIX 2: Added integer formatting specifications to block ALL trailing zeros completely
styled_col_info = (col_info.style
                   .apply(style_anomalies, axis=1)
                   .format({
                       'null_%': '{:.2f}%',
                       'null_count': '{:d}',
                       'n_unique': '{:d}'
                   }))

display(styled_col_info)
print()

# Print terminal/console notifications
has_issues = False

if hidden_numeric_cols:
    print(f"• Hidden Numeric Mapping Found:")
    print(f"  → {', '.join(hidden_numeric_cols)}: stored as str/object but should be numeric\n")
    has_issues = True

if dirty_categorical_cols:
    print(f"• Formatting Anomalies Found:")
    for col, vals in dirty_categorical_cols:
        print(f"  → {col}: contains dirty categorical values {vals}")
    print()
    has_issues = True

if not has_issues:
    print("✓ Every datatype and categorical entry is clean and correct!\n")


Shape: (400, 26)  →  400 records, 26 features



,pandas_dtype,null_count,null_%,n_unique,sample_values
rbc,object,152,38.00%,2,"['normal', 'abnormal']"
rc,object,130,32.50%,49,"['5.2', '3.9', '4.6']"
wc,object,105,26.25%,92,"['7800', '6000', '7500']"
pot,float64,88,22.00%,40,"[2.5, 3.2, 4.0]"
sod,float64,87,21.75%,34,"[111.0, 142.0, 104.0]"
pcv,object,70,17.50%,44,"['44', '38', '31']"
pc,object,65,16.25%,2,"['normal', 'abnormal']"
hemo,float64,52,13.00%,115,"[15.4, 11.3, 9.6]"
su,float64,49,12.25%,6,"[0.0, 3.0, 4.0]"
sg,float64,47,11.75%,5,"[1.02, 1.01, 1.005]"



• Hidden Numeric Mapping Found:
  → pcv, wc, rc: stored as str/object but should be numeric

• Formatting Anomalies Found:
  → dm: contains dirty categorical values ['yes', 'no', ' yes', '\tno', '\tyes']
  → cad: contains dirty categorical values ['no', 'yes', '\tno']
  → classification: contains dirty categorical values ['ckd', 'ckd\t', 'notckd']



In [4]:
df_clean = df_raw.copy()

print("--- Starting Automated Data Cleaning Pipeline ---")

# STEP 1: Fix Formatting Anomalies (Must be done first)
dirty_cols = [item[0] for item in dirty_categorical_cols] if dirty_categorical_cols else []

if dirty_cols:
    for col in dirty_cols:
        # UNIVERSAL FIX: Strip only if the value is a string, preserving real NaNs perfectly
        df_clean[col] = df_clean[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
        
        # Clean up specific dataset typos without corrupting valid data types
        df_clean[col] = df_clean[col].replace({'?': np.nan})
    print(f"✓ Successfully stripped whitespace and resolved formatting anomalies in: {dirty_cols}")
else:
    print("• No formatting anomalies required cleaning.")


# STEP 2: Fix Hidden Numeric Data Types (Done safely second)
if hidden_numeric_cols:
    for col in hidden_numeric_cols:
        # UNIVERSAL FIX: Strip strings safely without converting native NaNs to text
        df_clean[col] = df_clean[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
        
        # Remove character typos before numeric casting
        df_clean[col] = df_clean[col].replace({'?': np.nan})
        
        # Convert to actual numeric layout safely (forces any remaining bad text to NaN)
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        
    print(f"✓ Successfully converted hidden object columns to numeric layout: {hidden_numeric_cols}")
else:
    print("• No hidden numeric conversions were required.")

print("\n--- Pipeline Complete! Verifying structural changes: ---")
# Safe verification check even if one of the lists is empty
cols_to_verify = hidden_numeric_cols + dirty_cols
if cols_to_verify:
    print(df_clean[cols_to_verify].dtypes)
else:
    print("No columns needed modification.")


--- Starting Automated Data Cleaning Pipeline ---
✓ Successfully stripped whitespace and resolved formatting anomalies in: ['dm', 'cad', 'classification']
✓ Successfully converted hidden object columns to numeric layout: ['pcv', 'wc', 'rc']

--- Pipeline Complete! Verifying structural changes: ---
pcv               float64
wc                float64
rc                float64
dm                 object
cad                object
classification     object
dtype: object


In [5]:
df_clean = df_clean.drop('id', axis=1)
df_clean.shape

(400, 25)

In [6]:
df = df_clean

print(f"=== Post-Cleaning Data Quality Validation Target: [df] ===")
print(f'Shape: {df.shape}  →  {df.shape[0]} records, {df.shape[1]} features\n')

# High-performance column inspection on df
col_info_validated = pd.DataFrame({
    'pandas_dtype' : df.dtypes.astype(str),
    'null_count'   : df.isna().sum(),
    'null_%'       : (df.isna().mean() * 100).round(2),
    'n_unique'     : df.nunique(),
    'sample_values': [str(df[c].dropna().unique()[:3].tolist()) for c in df.columns]
}).sort_values('null_%', ascending=False)


def scan_data_quality(dataframe):
    hidden_numeric = []
    dirty_categorical = []
    obj_df = dataframe.select_dtypes(include=['object', 'string'])
    
    if obj_df.empty:
        return hidden_numeric, dirty_categorical
    
    for col in obj_df.columns:
        unique_vals = obj_df[col].dropna().unique()
        if unique_vals.size == 0:
            continue
            
        converted = pd.to_numeric(pd.Series(unique_vals), errors='coerce')
        is_probably_id = dataframe[col].nunique() == len(dataframe)
        
        if converted.notna().mean() > 0.5 and not is_probably_id:
            hidden_numeric.append(col)
            continue
            
        unique_series = pd.Series(unique_vals).astype(str)
        if unique_series.size != unique_series.str.strip().nunique():
            dirty_categorical.append((col, unique_vals.tolist()))
            
    return hidden_numeric, dirty_categorical


# Execute data scan on df
hidden_numeric_cols, dirty_categorical_cols = scan_data_quality(df)
dirty_cols_names = [item[0] for item in dirty_categorical_cols]

# Define the row-by-row styling function for the DataFrame
def style_anomalies(row):
    color_numeric = 'background-color: #ffcccc'      # Soft red/coral
    color_categorical = 'background-color: #fff2cc'  # Soft yellow/gold
    color_default = ''                                # Standard transparent
    
    if row.name in hidden_numeric_cols:
        return [color_numeric] * len(row)
    elif row.name in dirty_cols_names:
        return [color_categorical] * len(row)
    return [color_default] * len(row)

# Render colored table and format floats cleanly without trailing zeros
styled_col_info_validated = (col_info_validated.style
                             .apply(style_anomalies, axis=1)
                             .format({
                                 'null_%': '{:.2f}%',
                                 'null_count': '{:d}',
                                 'n_unique': '{:d}'
                             }))

display(styled_col_info_validated)
print()

# Print terminal/console notifications
has_issues = False

if hidden_numeric_cols:
    print(f"• Hidden Numeric Mapping Found:")
    print(f"  → {', '.join(hidden_numeric_cols)}: stored as str/object but should be numeric\n")
    has_issues = True

if dirty_categorical_cols:
    print(f"• Formatting Anomalies Found:")
    for col, vals in dirty_categorical_cols:
        print(f"  → {col}: contains dirty categorical values {vals}")
    print()
    has_issues = True

if not has_issues:
    print("✓ Every datatype and categorical entry is clean and correct!\n")


=== Post-Cleaning Data Quality Validation Target: [df] ===
Shape: (400, 25)  →  400 records, 25 features



,pandas_dtype,null_count,null_%,n_unique,sample_values
rbc,object,152,38.00%,2,"['normal', 'abnormal']"
rc,float64,131,32.75%,45,"[5.2, 3.9, 4.6]"
wc,float64,106,26.50%,89,"[7800.0, 6000.0, 7500.0]"
pot,float64,88,22.00%,40,"[2.5, 3.2, 4.0]"
sod,float64,87,21.75%,34,"[111.0, 142.0, 104.0]"
pcv,float64,71,17.75%,42,"[44.0, 38.0, 31.0]"
pc,object,65,16.25%,2,"['normal', 'abnormal']"
hemo,float64,52,13.00%,115,"[15.4, 11.3, 9.6]"
su,float64,49,12.25%,6,"[0.0, 3.0, 4.0]"
sg,float64,47,11.75%,5,"[1.02, 1.01, 1.005]"



✓ Every datatype and categorical entry is clean and correct!



In [7]:
import pandas as pd

# --- 1. Checking for Duplicate Rows ---
duplicate_row_count = df.duplicated().sum()

print("--- Checking for Duplicate Rows ---")
if duplicate_row_count > 0:
    df = df.drop_duplicates(keep='first').reset_index(drop=True)
    print(f"✓ Removed {duplicate_row_count} duplicate rows.")
else:
    print("✓ Clean! No duplicate rows detected.")

print(f"Current Shape: {df.shape[0]} rows, {df.shape[1]} features\n")


# --- 2. Safe & Scalable Duplicate Column Investigation ---
print("--- Investigating Duplicate Columns ---")
seen_columns = {}
duplicate_groups = {}

# Loop through columns and hash data to map exact duplicates
for col in df.columns:
    # Convert column content to a tuple to make it hashable
    # Using .tolist() handles unhashable nested types more safely
    col_hash = tuple(df[col].tolist())
    
    if col_hash in seen_columns:
        original_col = seen_columns[col_hash]
        if original_col not in duplicate_groups:
            duplicate_groups[original_col] = []
        duplicate_groups[original_col].append(col)
    else:
        seen_columns[col_hash] = col

# Print the findings clearly
if duplicate_groups:
    print(f"⚠ Found {len(duplicate_groups)} group(s) of identical columns:")
    for original, duplicates in duplicate_groups.items():
        print(f"  • Original column: '{original}' has identical twin(s): {duplicates}")
    print("\nℹ Action: No columns were deleted. Please review the twins listed above.")
else:
    print("✓ Clean! No duplicate columns detected.")

print(f"Final Shape: {df.shape[0]} rows, {df.shape[1]} features\n")

--- Checking for Duplicate Rows ---
✓ Clean! No duplicate rows detected.
Current Shape: 400 rows, 25 features

--- Investigating Duplicate Columns ---
✓ Clean! No duplicate columns detected.
Final Shape: 400 rows, 25 features



In [8]:
df['classification'].unique()

array(['ckd', 'notckd'], dtype=object)

In [9]:
target_mapping = {'ckd': 1, 'notckd': 0}
df['classification'] = df['classification'].map(target_mapping).astype(int)

In [10]:
df['classification'].unique()

array([1, 0])

In [11]:
# 1. Automatically extract quantitative (numerical) features
# Includes float64, int64, float32, etc.
quan = df.select_dtypes(include=['number']).columns.tolist()

# 2. Automatically extract qualitative (categorical) features
# Includes object, category, and boolean flags
qual = df.select_dtypes(exclude=['number']).columns.tolist()

print(f"🔢 Quantitative (Numerical) Columns ({len(quan)}): {quan}")
print(f"🔠 Qualitative (Categorical) Columns ({len(qual)}): {qual}")

🔢 Quantitative (Numerical) Columns (15): ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'classification']
🔠 Qualitative (Categorical) Columns (10): ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']


In [12]:
quan.remove('classification')

In [13]:
print(f"🔢 Quantitative (Numerical) Columns ({len(quan)}): {quan}")
print(f"🔠 Qualitative (Categorical) Columns ({len(qual)}): {qual}")

🔢 Quantitative (Numerical) Columns (14): ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc']
🔠 Qualitative (Categorical) Columns (10): ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']


In [14]:
X = df.drop(columns=['classification'])
y = df['classification']

# Splitting

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Training Set Size: {X_train.shape[0]} records")
print(f"Testing Set Size:  {X_test.shape[0]} records\n")

Training Set Size: 320 records
Testing Set Size:  80 records



In [16]:
def audit_missing_data(dataframe):
    """
    High-performance vectorized missing data audit.
    Bypasses row-by-row apply loops using pandas categorical cutting.
    """
    # 1. Native vectorized computations
    null_count = dataframe.isna().sum()
    null_pct = (dataframe.isna().mean() * 100).round(2)
    
    # 2. Optimized vectorized binning
    bins =  [-1,   0,       5,          15,          30,        50,       100]
    labels = [
        '✅ COMPLETE', 
        '✅ VERY LOW', 
        '🟢 LOW', 
        '🟡 MODERATE', 
        '🟠 HIGH', 
        '🔴 DROP'
    ]
    
    tiers = pd.cut(null_pct, bins=bins, labels=labels, right=True).astype(str)
    
    # 3. Construct dense audit summary grid
    audit_df = pd.DataFrame({
        'Missing Count': null_count,
        'Missing %': null_pct,
        'dtype': dataframe.dtypes.astype(str),
        'Tier': tiers
    }).sort_values('Missing %', ascending=False)
    
    return audit_df


def extract_imputation_lists(dataframe, audit_df):
    """
    Categorizes features into specific lists based on datatype and 
    missingness percentage thresholds to facilitate customized imputation.
    """
    # Separate columns into broad datatype masks
    # (Accounting for different variants of numerical types like int64, float64)
    num_cols = dataframe.select_dtypes(include=['number']).columns
    cat_cols = dataframe.select_dtypes(exclude=['number']).columns
    
    # Extract index series from the audit dataframe for easy filtering
    pct = audit_df['Missing %']
    
    # Generate the precise lists based on your thresholds
    # Using strict inequalities to ensure no feature overlaps across groups
    drop_col = audit_df[pct > 50].index.tolist()
    
    high_num = audit_df[(pct > 30) & (pct <= 50) & (audit_df.index.isin(num_cols))].index.tolist()
    high_cate = audit_df[(pct > 30) & (pct <= 50) & (audit_df.index.isin(cat_cols))].index.tolist()
    
    mid_num = audit_df[(pct > 15) & (pct <= 30) & (audit_df.index.isin(num_cols))].index.tolist()
    mid_cate = audit_df[(pct > 15) & (pct <= 30) & (audit_df.index.isin(cat_cols))].index.tolist()
    
    low_num = audit_df[(pct > 5) & (pct <= 15) & (audit_df.index.isin(num_cols))].index.tolist()
    low_cate = audit_df[(pct > 5) & (pct <= 15) & (audit_df.index.isin(cat_cols))].index.tolist()
    
    very_low_num = audit_df[(pct <= 5) & (audit_df.index.isin(num_cols))].index.tolist()
    very_low_cate = audit_df[(pct <= 5) & (audit_df.index.isin(cat_cols))].index.tolist()
    
    # Package into a clean dictionary configuration
    lists_dict = {
        "drop_col": drop_col,
        "high_num": high_num,
        "high_cate": high_cate,
        "mid_num": mid_num,
        "mid_cate": mid_cate,
        "low_num": low_num,
        "low_cate": low_cate,
        "very_low_num": very_low_num,
        "very_low_cate": very_low_cate
    }
    
    return lists_dict


# ========================================================
# EXECUTION
# ========================================================

# 1. Run the audit
training_audit = audit_missing_data(X_train)
print('📊 Training Split Automated Missing Value Audit:')
print('=' * 65)
print(training_audit.to_string())

# 2. Extract your lists
imputation_configs = extract_imputation_lists(X_train, training_audit)

# 3. Unpack them into individual variables so you can use them easily
drop_col = imputation_configs["drop_col"]
high_num = imputation_configs["high_num"]
high_cate = imputation_configs["high_cate"]
mid_num = imputation_configs["mid_num"]
mid_cate = imputation_configs["mid_cate"]
low_num = imputation_configs["low_num"]
low_cate = imputation_configs["low_cate"]
very_low_num = imputation_configs["very_low_num"]
very_low_cate = imputation_configs["very_low_cate"]

# Print confirmation of extracted lists
print('\n🎯 Extracted Feature Lists for Imputation:')
print('=' * 65)
for key, feature_list in imputation_configs.items():
    print(f"🔹 {key:14} (Count: {len(feature_list)}): {feature_list}")


📊 Training Split Automated Missing Value Audit:
       Missing Count  Missing %    dtype        Tier
rbc              126      39.38   object      🟠 HIGH
rc               105      32.81  float64      🟠 HIGH
wc                84      26.25  float64  🟡 MODERATE
pot               76      23.75  float64  🟡 MODERATE
sod               75      23.44  float64  🟡 MODERATE
pcv               58      18.12  float64  🟡 MODERATE
pc                54      16.88   object  🟡 MODERATE
hemo              43      13.44  float64       🟢 LOW
su                42      13.12  float64       🟢 LOW
sg                41      12.81  float64       🟢 LOW
al                40      12.50  float64       🟢 LOW
bgr               38      11.88  float64       🟢 LOW
bu                19       5.94  float64       🟢 LOW
sc                16       5.00  float64  ✅ VERY LOW
bp                11       3.44  float64  ✅ VERY LOW
age                8       2.50  float64  ✅ VERY LOW
pcc                4       1.25   object  ✅ VERY LO

In [17]:
descriptive = pd.DataFrame(
    index=[
        "Mean", "Median", "Mode", "Q1:25%", "Q2:50%", "Q3:75%", "Q4:100%", 
        "IQR", "1.5rule", "Lesser", "Greater", "Min", "Max", "Skewness", "Kurtosis"
    ], 
    columns=quan
)

# 2. Populate the metrics looping through your quantitative features
# Note: Ensure 'df' here represents your training set (e.g., df = X_train) to prevent data leakage!
for columnName in quan:
    descriptive.loc["Mean", columnName] = df[columnName].mean()
    descriptive.loc["Median", columnName] = df[columnName].median()
    descriptive.loc["Mode", columnName] = df[columnName].mode()[0]
    
    # Using .loc on df.describe() as well to keep it consistent and warning-free
    desc = df[columnName].describe()
    descriptive.loc["Q1:25%", columnName] = desc["25%"]
    descriptive.loc["Q2:50%", columnName] = desc["50%"]
    descriptive.loc["Q3:75%", columnName] = desc["75%"]
    descriptive.loc["Q4:100%", columnName] = desc["max"]
    
    descriptive.loc["IQR", columnName] = descriptive.loc["Q3:75%", columnName] - descriptive.loc["Q1:25%", columnName]
    descriptive.loc["1.5rule", columnName] = 1.5 * descriptive.loc["IQR", columnName]
    descriptive.loc["Lesser", columnName] = descriptive.loc["Q1:25%", columnName] - descriptive.loc["1.5rule", columnName]
    descriptive.loc["Greater", columnName] = descriptive.loc["Q3:75%", columnName] + descriptive.loc["1.5rule", columnName]
    
    descriptive.loc["Min", columnName] = df[columnName].min()
    descriptive.loc["Max", columnName] = df[columnName].max()
    descriptive.loc["Skewness", columnName] = df[columnName].skew()
    descriptive.loc["Kurtosis", columnName] = df[columnName].kurtosis()

In [18]:
descriptive

,age,bp,sg,al,su,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc
Mean,51.483376,76.469072,1.017408,1.016949,0.450142,148.036517,57.425722,3.072454,137.528754,4.627244,12.526437,38.884498,8406.122449,4.707435
Median,55.0,80.0,1.02,0.0,0.0,121.0,42.0,1.3,138.0,4.4,12.65,40.0,8000.0,4.8
Mode,60.0,80.0,1.02,0.0,0.0,99.0,46.0,1.2,135.0,3.5,15.0,41.0,9800.0,5.2
Q1:25%,42.0,70.0,1.01,0.0,0.0,99.0,27.0,0.9,135.0,3.8,10.3,32.0,6500.0,3.9
Q2:50%,55.0,80.0,1.02,0.0,0.0,121.0,42.0,1.3,138.0,4.4,12.65,40.0,8000.0,4.8
Q3:75%,64.5,80.0,1.02,2.0,0.0,163.0,66.0,2.8,142.0,4.9,15.0,45.0,9800.0,5.4
Q4:100%,90.0,180.0,1.025,5.0,5.0,490.0,391.0,76.0,163.0,47.0,17.8,54.0,26400.0,8.0
IQR,22.5,10.0,0.01,2.0,0.0,64.0,39.0,1.9,7.0,1.1,4.7,13.0,3300.0,1.5
1.5rule,33.75,15.0,0.015,3.0,0.0,96.0,58.5,2.85,10.5,1.65,7.05,19.5,4950.0,2.25
Lesser,8.25,55.0,0.995,-3.0,0.0,3.0,-31.5,-1.95,124.5,2.15,3.25,12.5,1550.0,1.65


In [19]:
def OutlierDetector(Quan, descriptive_df, data_df):
    """
    Identifies low and high outliers, returning master tables with columns organized as:
    outlier_count -> outlier_% -> threshold -> outlier_range.
    """
    low_summary_data = []
    high_summary_data = []
    
    LesserOut = []
    GreaterOut = []
    
    total_rows = len(data_df)
    
    for columnName in Quan:
        lesser_threshold = descriptive_df.loc["Lesser", columnName]
        greater_threshold = descriptive_df.loc["Greater", columnName]
        
        # 1. Process Low Outliers
        low_vals = data_df[data_df[columnName] < lesser_threshold][columnName].tolist()
        if len(low_vals) > 0:
            LesserOut.append(columnName)
            low_min, low_max = min(low_vals), max(low_vals)
            range_str = f"[{low_min}]" if low_min == low_max else f"[{low_min} to {low_max}]"
            
            low_summary_data.append({
                "column_name": columnName,
                "outlier_count": len(low_vals),
                "outlier_%": f"{(len(low_vals) / total_rows) * 100:.2f}%",  # Moved up
                "lesser_threshold": lesser_threshold,
                "outlier_range": range_str
            })
            
        # 2. Process High Outliers
        high_vals = data_df[data_df[columnName] > greater_threshold][columnName].tolist()
        if len(high_vals) > 0:
            GreaterOut.append(columnName)
            high_min, high_max = min(high_vals), max(high_vals)
            range_str = f"[{high_min}]" if high_min == high_max else f"[{high_min} to {high_max}]"
            
            high_summary_data.append({
                "column_name": columnName,
                "outlier_count": len(high_vals),
                "outlier_%": f"{(len(high_vals) / total_rows) * 100:.2f}%",  # Moved up
                "greater_threshold": greater_threshold,
                "outlier_range": range_str
            })
            
    df_low = pd.DataFrame(low_summary_data)
    df_high = pd.DataFrame(high_summary_data)
    
    if not df_low.empty:
        df_low.set_index("column_name", inplace=True)
    if not df_high.empty:
        df_high.set_index("column_name", inplace=True)
        
    return df_low, df_high, LesserOut, GreaterOut


In [20]:
# ========================================================
# HOW TO RUN AND DISPLAY IT
# ========================================================
df_low_summary, df_high_summary, lesser_outliers, greater_outliers = OutlierDetector(quan, descriptive, df)

print("📉 LOW OUTLIERS TABLE PROFILE:")
display(df_low_summary) 

print("\n📈 HIGH OUTLIERS TABLE PROFILE:")
display(df_high_summary)

📉 LOW OUTLIERS TABLE PROFILE:


,outlier_count,outlier_%,lesser_threshold,outlier_range
column_name,,,,
age,10,2.50%,8.25,[2.0 to 8.0]
bp,5,1.25%,55.00,[50.0]
sod,15,3.75%,124.50,[4.5 to 124.0]
hemo,1,0.25%,3.25,[3.1]
pcv,1,0.25%,12.50,[9.0]



📈 HIGH OUTLIERS TABLE PROFILE:


,outlier_count,outlier_%,greater_threshold,outlier_range
column_name,,,,
bp,31,7.75%,95.00,[100.0 to 180.0]
su,61,15.25%,0.00,[1.0 to 5.0]
bgr,34,8.50%,259.00,[261.0 to 490.0]
bu,38,9.50%,124.50,[125.0 to 391.0]
sc,51,12.75%,5.65,[5.9 to 76.0]
sod,1,0.25%,152.50,[163.0]
pot,4,1.00%,6.55,[6.6 to 47.0]
wc,10,2.50%,14750.00,[14900.0 to 26400.0]
rc,1,0.25%,7.65,[8.0]


In [21]:
def CleanExtreamOutliers(data_df, floors=None, ceilings=None):
    """
    A single master tool to sanitize both low and high extreme outliers 
    using expert-defined domain rules.
    """
    cleaned_df = data_df.copy()
    initial_row_count = len(cleaned_df)
    
    BOLD = "\033[1m"
    RESET = "\033[0m"
    LIGHT_BG = "\033[43m\033[30m"
    GRAY_TEXT = "\033[90m"
    SUCCESS_TEXT = "\033[92m"
    
    print("=" * 65)
    print(f"{BOLD}🧹 EXECUTING FULL DOMAIN-DRIVEN DATA SANITIZATION{RESET}")
    print("=" * 65)
    
    # 1. Process Low Floors
    if floors:
        print(f"{BOLD}[PART 1: LOW FLOORS]{RESET}")
        for col, floor_val in floors.items():
            if col in cleaned_df.columns:
                failing = (cleaned_df[col] < floor_val) & (cleaned_df[col].notna())
                rows_to_drop = failing.sum()
                if rows_to_drop > 0:
                    cleaned_df = cleaned_df[(cleaned_df[col] >= floor_val) | (cleaned_df[col].isna())]
                    row_text = f" {BOLD}{col:<6}{RESET}{LIGHT_BG} | Dropped {rows_to_drop:<2} row(s) below biological floor ({floor_val})"
                    print(f"{LIGHT_BG}{row_text:<65}{RESET}")
                else:
                    print(f"{GRAY_TEXT} {BOLD}{col:<6}{RESET}{GRAY_TEXT} | No rows violated floor ({floor_val}){RESET}")
        print("-" * 65)

    # 2. Process High Ceilings
    if ceilings:
        print(f"{BOLD}[PART 2: HIGH CEILINGS]{RESET}")
        for col, ceiling_val in ceilings.items():
            if col in cleaned_df.columns:
                failing = (cleaned_df[col] > ceiling_val) & (cleaned_df[col].notna())
                rows_to_drop = failing.sum()
                if rows_to_drop > 0:
                    cleaned_df = cleaned_df[(cleaned_df[col] <= ceiling_val) | (cleaned_df[col].isna())]
                    row_text = f" {BOLD}{col:<6}{RESET}{LIGHT_BG} | Dropped {rows_to_drop:<2} row(s) above biological ceiling ({ceiling_val})"
                    print(f"{LIGHT_BG}{row_text:<65}{RESET}")
                else:
                    print(f"{GRAY_TEXT} {BOLD}{col:<6}{RESET}{GRAY_TEXT} | No rows violated ceiling ({ceiling_val}){RESET}")
        print("-" * 65)
                
    total_dropped = initial_row_count - len(cleaned_df)
    print(f"{BOLD}{SUCCESS_TEXT}✅ Sanitization Complete: Total rows removed = {total_dropped}{RESET}")
    print("=" * 65)
    
    return cleaned_df

In [22]:
low_floors = {
    "age": 0.0,
    "bp": 40.0,
    "sod": 100.0,
    "hemo": 2.5,
    "pcv": 8.0
}

high_ceilings = {
    "bp": 200.0,
    "su": 6.0,
    "bgr": 600.0,
    "bu": 450.0,
    "sc": 20.0,
    "sod": 170.0,
    "pot": 10.0,
    "wc": 30000.0,
    "rc": 9.0
}

# Run the tool
df = CleanExtreamOutliers(df, floors=low_floors, ceilings=high_ceilings)

🧹 EXECUTING FULL DOMAIN-DRIVEN DATA SANITIZATION
[PART 1: LOW FLOORS]
 age    | No rows violated floor (0.0)
 bp     | No rows violated floor (40.0)
 sod    | Dropped 1  row(s) below biological floor (100.0)
 hemo   | No rows violated floor (2.5)
 pcv    | No rows violated floor (8.0)
-----------------------------------------------------------------
[PART 2: HIGH CEILINGS]
 bp     | No rows violated ceiling (200.0)
 su     | No rows violated ceiling (6.0)
 bgr    | No rows violated ceiling (600.0)
 bu     | No rows violated ceiling (450.0)
 sc     | Dropped 3  row(s) above biological ceiling (20.0)
 sod    | No rows violated ceiling (170.0)
 pot    | Dropped 1  row(s) above biological ceiling (10.0)
 wc     | No rows violated ceiling (30000.0)
 rc     | No rows violated ceiling (9.0)
-----------------------------------------------------------------
✅ Sanitization Complete: Total rows removed = 5
